In [ ]:
import os
from pathlib import Path
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score

# Fix all random seeds for reproducibility across runs
def set_seed(seed=10879360):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(10879360)
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Locate the repository data; AV_DATA_DIR can override this on hosted runtimes.
candidates = [Path.cwd(), *Path.cwd().parents]
default_root = next((path for path in candidates if (path / 'data').is_dir()), Path.cwd())
REPO_ROOT = Path(os.getenv('AV_PROJECT_ROOT', default_root)).resolve()
DATA_DIR = Path(os.getenv('AV_DATA_DIR', REPO_ROOT / 'data'))

train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv')

# Verify dataset size, class distribution, and missing values
print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,}")
print(train_df['label'].value_counts())
print(train_df.isnull().sum())

In [ ]:
# Install requirements before running this notebook; see docs/reproduction.md.

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Remove email headers, addresses, and URLs to focus on authorship style
def clean_text(text):
    text = re.sub(r'(From|To|Cc|Subject|Date|Forwarded by)[^\n]*\n', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Tokenize text pairs and return tensors; label is optional for test inference
class AVDataset(Dataset):
    def __init__(self, df, has_label=True):
        df = df.copy()
        df['text_1'] = df['text_1'].apply(clean_text)
        df['text_2'] = df['text_2'].apply(clean_text)
        self.encodings = tokenizer(
            list(df['text_1']),
            list(df['text_2']),
            max_length=MAX_LENGTH,
            truncation='longest_first',
            padding=False,
            return_tensors=None
        )
        self.labels = df['label'].values if has_label else None
        self.has_label = has_label

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
        }
        if self.has_label:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataset = AVDataset(train_df)
dev_dataset = AVDataset(dev_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2,
                          collate_fn=data_collator)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2,
                        collate_fn=data_collator)

print(f"Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}")

In [ ]:
OUTPUT_DIR = Path(os.getenv('AV_MODEL_DIR', REPO_ROOT / 'models' / 'roberta-asymmetric-loss'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 14
LEARNING_RATE = 5e-6

# Asymmetric Loss: applies stronger focal penalty to hard negatives (gamma_neg > gamma_pos)
class ASLoss(nn.Module):
    def __init__(self, gamma_neg=2, gamma_pos=1):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos

    def forward(self, logits, labels):
        probs = torch.softmax(logits, dim=-1)
        probs_pos = probs[:, 1]
        probs_neg = probs[:, 0]
        los_pos = labels * torch.log(probs_pos.clamp(min=1e-8))
        los_neg = (1 - labels) * torch.log(probs_neg.clamp(min=1e-8))
        loss = los_pos + los_neg
        loss[labels == 1] *= (1 - probs_pos[labels == 1]) ** self.gamma_pos
        loss[labels == 0] *= (1 - probs_neg[labels == 0]) ** self.gamma_neg
        return -loss.mean()

# Custom Trainer that substitutes ASL for the default cross-entropy loss
class AVTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = ASLoss(gamma_neg=2, gamma_pos=1)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    seed=10879360,
    save_only_model=True
)

print(f"Training setup complete!")

In [ ]:
import json
from sklearn.metrics import f1_score
import numpy as np

# Load pretrained RoBERTa-Large with a binary classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32
)

trainer = AVTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()
print(f"\nBest Val F1: {trainer.state.best_metric:.4f}")

# Sweep thresholds on dev set to find the value that maximises macro F1
def find_best_threshold(trainer, dataset):
    predictions = trainer.predict(dataset)
    probs = torch.softmax(
        torch.tensor(predictions.predictions), dim=-1
    )[:, 1].numpy()
    labels = predictions.label_ids

    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.2, 0.8, 0.01):
        preds = (probs >= thresh).astype(int)
        f1 = f1_score(labels, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    print(f"Best threshold: {best_thresh:.2f} | F1: {best_f1:.4f}")
    return best_thresh

best_threshold = find_best_threshold(trainer, dev_dataset)

# Export the best model and tokenizer into the shared inference directory.
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# Save optimal threshold alongside model weights for inference
threshold_data = {'best_threshold': float(best_threshold)}
with open(OUTPUT_DIR / 'best_threshold.json', 'w') as f:
    json.dump(threshold_data, f)

print(f"Model saved at: {OUTPUT_DIR}")
print(f"Best threshold saved: {best_threshold:.2f}")